# Task 2:Support Vector Machine(SVM) Classifier

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,precision_score,recall_score,roc_auc_score
from sklearn.decomposition import PCA


## Loading the dataset

In [ ]:
# load the dataset
df = pd.read_csv(r"C:/Users/Hp/Documents/My project/Codveda Projects/Dataset/Churn Prdiction Data/churn-bigml-20.csv")
df.head()
df.info()



## Handling dataset

In [ ]:
# handling dataset
print("Missing values before handling:", df.isnull().sum())
print("Duplicate before handling:",df.duplicated().sum())

## Encoding Catagorical Features

In [ ]:
#encoding
if "State" in df.columns:
  df=pd.get_dummies(df,columns=["State"],drop_first=True)

df["International plan"] = df["International plan"].replace({"No":0,"Yes":1})
df["Voice mail plan"] = df["Voice mail plan"].replace({"No":0,"Yes":1})
df["Churn"] = df["Churn"].replace({False:0,True:1})
df.head()


## Feature and Target Separation

In [ ]:
# feature and target separation
X=df.drop(columns=["Churn"],axis=1,errors="ignore")
y=df["Churn"]

## Train Test Split and Feature Scaling

In [ ]:
# Train-Test split and  feature scaling
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=True,random_state=42)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

## SVM – Linear Kernel

Linear SVM finds a straight hyperplane to separate churn and non-churn classes. It works best when data is approximately linearly separable. The model is trained using scaled features and balanced class weights to handle imbalance.

In [ ]:
#Linear Kernel SVM
svm_linear = SVC(kernel='linear', probability=True, class_weight="balanced")
svm_linear.fit(X_train, y_train)

y_pred_linear = svm_linear.predict(X_test)
y_prob_linear = svm_linear.predict_proba(X_test)[:, 1]

## SVM – RBF Kernel

RBF SVM uses a non-linear kernel to map data into higher dimensions and capture complex patterns. It is suitable when data is not linearly separable. Gamma is set to 'scale' and class imbalance is handled using class weights.

In [ ]:
# RBF Kernel SVM
svm_rbf = SVC(kernel='rbf', probability=True, gamma='scale',class_weight="balanced")
svm_rbf.fit(X_train, y_train)

y_pred_rbf = svm_rbf.predict(X_test)
y_prob_rbf = svm_rbf.predict_proba(X_test)[:, 1]

## Model Evaluation

Models are evaluated using Accuracy, Precision, Recall, and AUC. Since the dataset is imbalanced, Recall and AUC are more important than Accuracy. A comparison function is used to store results in a structured format.

In [ ]:
#model evaluation
def evaluate_model(y_test, y_pred, y_prob, name):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
    }

linear_results=evaluate_model(y_test, y_pred_linear, y_prob_linear, "Linear SVM")
rbf_results=evaluate_model(y_test, y_pred_rbf, y_prob_rbf, "RBF SVM")
print(linear_results)
print(rbf_results)

## Model Comparison

Linear and RBF SVM models are compared using evaluation metrics. The best model is chosen based on Recall and AUC rather than Accuracy, as churn datasets are imbalanced.

In [ ]:
#comparision table
results_df=pd.DataFrame([linear_results,rbf_results])
print(results_df)

## Decision Boundary Visualization

PCA reduces data to 2D for visualization. An SVM model is trained on this reduced data to plot decision boundaries. The plot shows how the model separates churn and non-churn regions. This is only for visualization, not final evaluation.

In [ ]:
#visualization of the decision boundary
# Reduce to 2D using PCA
pca = PCA(n_components=2)
X_train_pca=pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)


#Train SVM on ONLY 2D data
model = SVC(kernel='rbf')
model.fit(X_train_pca, y_train)

#plot decision boundary
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, edgecolors='k', cmap='coolwarm')
plt.title("SVM Decision Boundary (RBF Kernel)")
plt.show()

## Conclusion

Linear and RBF SVM models were compared for churn prediction. Although RBF achieved higher accuracy, Linear SVM performed better in recall and provided more reliable churn detection. Therefore, Linear SVM is more suitable for this dataset.